# Supervised Model Comparison: Random Forest vs. XGBoost

Week 5 deliverable — compares the two supervised baselines on the held-out test set
(Jan 2024 onward) across three threat labels: fire, drought, and vegetation degradation.

**Sections**
1. Overall test performance — F2, precision, recall, Brier score, bootstrap 95% CIs
2. Per-park breakdown — F2 heatmap across six Nigerian national parks
3. Calibration diagnostics — reliability diagrams for both models
4. Lead-time analysis — mean days of advance warning per threat

## Metrics Reference

All metrics are computed on the held-out test set (Jan 2024 onward) at a classification threshold of 0.50 unless noted.

| Metric | What it measures | Range | Better when |
|---|---|---|---|
| **F2** | Weighted harmonic mean of precision and recall, with recall weighted 2× — the **primary metric** | 0–1 | Higher |
| **Precision** | Of all positive predictions, what fraction are correct | 0–1 | Higher |
| **Recall** | Of all actual positive events, what fraction did the model detect | 0–1 | Higher |
| **Brier score** | Mean squared error between predicted probability and true label | 0–1 | Lower |
| **ROC-AUC** | Probability that the model ranks a positive above a negative; threshold-independent | 0–1 | Higher |
| **95% CI** | Bootstrap confidence interval for F2 (1,000 non-parametric resamples) | — | Narrow |
| **Persist F2** | F2 of the persistence baseline: predict today's label as tomorrow's label | 0–1 | — |
| **Beats Baseline** | Whether the model's F2 exceeds the persistence baseline | Yes/No | Yes |
| **Lead time** | Mean days of advance warning before a positive event onset, at P ≥ 0.70 | days | Higher |

**Why F2 over F1?** In conservation, failing to detect a threat (false negative) carries a higher cost than a false alarm that prompts unnecessary inspection (false positive). F2 penalises missed detections more heavily than F1.

In [ ]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.calibration import calibration_curve

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / "src" / "models"))

from model_utils import LABEL_COLS, apply_platt, load_splits, prep_arrays
from evaluate import THREAT_LABELS

RESULTS = ROOT / "results"
MODELS  = RESULTS / "models"
(RESULTS / "plots").mkdir(parents=True, exist_ok=True)

THREAT_SHORT = {
    "fire_within_30d":       "Fire",
    "drought_within_30d":    "Drought",
    "vegetation_within_30d": "Vegetation",
}
PARK_LABELS = {
    "chad_basin":    "Chad Basin",
    "cross_river":   "Cross River",
    "gashaka_gumti": "Gashaka-Gumti",
    "kainji":        "Kainji",
    "old_oyo":       "Old Oyo",
    "yankari":       "Yankari",
}
MODEL_COLORS = {"RF": "#2196F3", "XGBoost": "#FF5722"}

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
print("Setup complete. ROOT:", ROOT)

In [ ]:
with open(RESULTS / "rf_supervised_results.json") as f:
    rf_res = json.load(f)
with open(RESULTS / "xgb_supervised_results.json") as f:
    xgb_res = json.load(f)

print(f"RF  best params: {rf_res['best_params']}")
print(f"XGB best params: {xgb_res['best_params']}")
print(f"RF  test mean-F2: {rf_res['test_mean_f2']:.4f}")
print(f"XGB test mean-F2: {xgb_res['test_mean_f2']:.4f}")

## 1. Overall Test Performance

F2 (beta=2) is the primary metric — it weights recall twice as heavily as precision,
reflecting the cost asymmetry: missing a real threat is worse than a false alarm
in conservation settings. Bootstrap 95% CIs use 1,000 non-parametric resamples.

In [ ]:
rows = []
for label in THREAT_LABELS:
    for model_name, res in [("RF", rf_res), ("XGBoost", xgb_res)]:
        m  = res["test_metrics"][label]
        ci = m["bootstrap_f2"]
        rows.append({
            "Threat":         THREAT_SHORT[label],
            "Model":          model_name,
            "F2":             round(m["f2"], 4),
            "95% CI":         f"[{ci['ci_lower']:.4f}, {ci['ci_upper']:.4f}]",
            "Precision":      round(m["precision"], 4),
            "Recall":         round(m["recall"], 4),
            "Brier":          round(m["brier"], 4),
            "ROC-AUC":        round(m["roc_auc"], 4),
            "Persist F2":     round(m["persistence_f2"], 4),
            "Beats Baseline": "Yes" if m["beats_baseline"] else "No",
        })

df_compare = pd.DataFrame(rows).set_index(["Threat", "Model"])
df_compare

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

for ax, label in zip(axes, THREAT_LABELS):
    for j, (model_name, res) in enumerate([("RF", rf_res), ("XGBoost", xgb_res)]):
        m  = res["test_metrics"][label]
        ci = m["bootstrap_f2"]
        f2 = m["f2"]
        lo = f2 - ci["ci_lower"]
        hi = ci["ci_upper"] - f2
        ax.bar(j, f2, color=MODEL_COLORS[model_name], alpha=0.85, width=0.5)
        ax.errorbar(j, f2, yerr=[[lo], [hi]], fmt="none",
                    color="black", capsize=5, linewidth=1.5)
        ax.text(j, ci["ci_upper"] + 0.015, f"{f2:.3f}",
                ha="center", va="bottom", fontsize=9, fontweight="bold")

    persist = rf_res["test_metrics"][label]["persistence_f2"]
    ax.axhline(persist, color="gray", linestyle="--", linewidth=1)
    ax.set_title(THREAT_SHORT[label], fontweight="bold")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["RF", "XGBoost"])
    ax.set_ylim(0, 1.12)
    if ax is axes[0]:
        ax.set_ylabel("F2 Score")

handles = [
    mpatches.Patch(color=MODEL_COLORS["RF"],      label="Random Forest"),
    mpatches.Patch(color=MODEL_COLORS["XGBoost"], label="XGBoost"),
    plt.Line2D([0], [0], color="gray", linestyle="--", label="Persistence baseline"),
]
fig.legend(handles=handles, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.05))
fig.suptitle("F2 Score on Test Set (Jan 2024 onward) — 95% Bootstrap CI",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(RESULTS / "plots" / "f2_comparison.png", bbox_inches="tight")
plt.show()

## 2. Per-Park Breakdown

F2 scores broken down by park and threat. Values below 0.40 appear in red; above 0.80 in green.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for ax, (model_name, res) in zip(axes, [("RF", rf_res), ("XGBoost", xgb_res)]):
    parks = list(res["per_park_test"].keys())
    threat_labels_short = [THREAT_SHORT[l] for l in THREAT_LABELS]
    data = np.array([
        [res["per_park_test"][p][l]["f2"] for l in THREAT_LABELS]
        for p in parks
    ])
    park_display = [PARK_LABELS.get(p, p) for p in parks]

    im = ax.imshow(data, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(3))
    ax.set_xticklabels(threat_labels_short)
    ax.set_yticks(range(len(parks)))
    ax.set_yticklabels(park_display)
    ax.set_title(f"{model_name} — Per-Park F2", fontweight="bold")

    for i in range(len(parks)):
        for j in range(3):
            v = data[i, j]
            text_color = "white" if v < 0.35 or v > 0.85 else "black"
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color=text_color, fontsize=9, fontweight="bold")

    plt.colorbar(im, ax=ax, label="F2 Score")

plt.tight_layout()
plt.savefig(RESULTS / "plots" / "per_park_f2_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
park_rows = []
for model_name, res in [("RF", rf_res), ("XGBoost", xgb_res)]:
    for park, pm in res["per_park_test"].items():
        row = {"Model": model_name, "Park": PARK_LABELS.get(park, park)}
        for label in THREAT_LABELS:
            row[THREAT_SHORT[label]] = round(pm[label]["f2"], 3)
        park_rows.append(row)

df_park = pd.DataFrame(park_rows).set_index(["Model", "Park"])
df_park

## 3. Calibration Diagnostics

Reliability diagrams compare predicted probabilities against observed event frequencies.
Points on the diagonal indicate perfect calibration.
Both models use Platt scaling fitted on the validation set; the vegetation scaler
fell back to raw probabilities due to a negative coefficient (seasonal distribution
mismatch in the Jul-Dec 2023 validation window).

In [ ]:
_, _, test_df = load_splits()

with open(MODELS / "rf_supervised" / "model.pkl", "rb") as f:
    rf_bundle = pickle.load(f)
with open(MODELS / "xgb_supervised" / "model.pkl", "rb") as f:
    xgb_bundle = pickle.load(f)

X_test, Y_test, _ = prep_arrays(test_df, rf_bundle["imputer"])

rf_probs_raw  = np.column_stack(
    [p[:, 1] for p in rf_bundle["model"].predict_proba(X_test)]
)
rf_probs = apply_platt(rf_probs_raw, rf_bundle["calibrators"])

xgb_probs_raw = np.column_stack(
    [m.predict_proba(X_test)[:, 1] for m in xgb_bundle["models"]]
)
xgb_probs = apply_platt(xgb_probs_raw, xgb_bundle["calibrators"])

print("Test rows:", X_test.shape[0])
for i, label in enumerate(THREAT_LABELS):
    print(f"  {THREAT_SHORT[label]:12s}  RF [{rf_probs[:,i].min():.3f}, {rf_probs[:,i].max():.3f}]  "
          f"XGB [{xgb_probs[:,i].min():.3f}, {xgb_probs[:,i].max():.3f}]")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

for ax, (i, label) in zip(axes, enumerate(THREAT_LABELS)):
    y_true_raw = Y_test[:, i].astype(float)
    valid  = ~np.isnan(y_true_raw)
    y_true = y_true_raw[valid].astype(int)

    for model_name, probs, color in [
        ("RF",      rf_probs,  MODEL_COLORS["RF"]),
        ("XGBoost", xgb_probs, MODEL_COLORS["XGBoost"]),
    ]:
        yp = probs[valid, i]
        frac, mean_pred = calibration_curve(y_true, yp, n_bins=10, strategy="uniform")
        ax.plot(mean_pred, frac, marker="o", markersize=4,
                color=color, label=model_name, linewidth=1.5)

    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect")
    ax.set_title(THREAT_SHORT[label], fontweight="bold")
    ax.set_xlabel("Mean predicted probability")
    if i == 0:
        ax.set_ylabel("Fraction of positives")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=9)

fig.suptitle("Reliability Diagrams — Calibrated Probabilities vs. Observed Frequency",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(RESULTS / "plots" / "reliability_diagrams.png", bbox_inches="tight")
plt.show()

## 4. Lead-Time Analysis

Lead time is the mean number of days before each positive-event onset that the predicted
probability first crosses 0.70. A value of 0 means the model only fires on or after the event starts.

In [ ]:
lt_rows = []
for label in THREAT_LABELS:
    for model_name, res in [("RF", rf_res), ("XGBoost", xgb_res)]:
        lt_rows.append({
            "Threat":           THREAT_SHORT[label],
            "Model":            model_name,
            "Lead Time (days)": round(res["test_metrics"][label]["lead_time_days"], 2),
        })

df_lt = pd.DataFrame(lt_rows)
print(df_lt.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3)
w = 0.30

for j, (model_name, res) in enumerate([("RF", rf_res), ("XGBoost", xgb_res)]):
    vals = [res["test_metrics"][l]["lead_time_days"] for l in THREAT_LABELS]
    bars = ax.bar(x + j * w, vals, w, color=MODEL_COLORS[model_name], alpha=0.85, label=model_name)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                f"{v:.1f}d", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x + w / 2)
ax.set_xticklabels([THREAT_SHORT[l] for l in THREAT_LABELS])
ax.set_ylabel("Mean lead time (days)")
ax.set_title("Early-Warning Lead Time at P ≥ 0.70 Threshold", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS / "plots" / "lead_time_comparison.png", bbox_inches="tight")
plt.show()

## Summary

| Model         | Mean F2 | Fire F2 | Drought F2 | Vegetation F2 |
|---|---|---|---|---|
| Random Forest | 0.8565  | 0.9436  | 0.8818     | 0.7441        |
| XGBoost       | 0.8267  | 0.9355  | 0.8415     | 0.7030        |
| Persistence   | —       | 0.7798  | 0.3991     | 0.4796        |

Both models beat the persistence baseline on all three threats.
Random Forest achieves a higher mean F2 (0.8565 vs. 0.8267) and is the primary
supervised baseline carried into Weeks 6 and 7.

**Key observations**
- Fire detection is strongest (F2 > 0.93) — fire history features and NDVI provide a clear signal.
- Drought benefits from rain-deficit and temperature accumulators (F2 > 0.84 for RF).
- Vegetation degradation is the hardest task (F2 approximately 0.70-0.74); NDVI-derived labels
  introduce potential leakage, flagged as a thesis limitation.
- Calibration is good for fire and drought. Vegetation shows deviation due to seasonal
  distribution mismatch in the validation window (Jul-Dec 2023, rainy season),
  which caused the Platt scaler to learn an inverted coefficient; raw probabilities
  were preserved via passthrough logic.